# ฝึก AI ตรวจจับความผิดปกติของมอเตอร์ (Colab)
โน้ตบุ๊กนี้ทำงานเหมือน `python/train.py` แต่แสดงทีละขั้นพร้อมกราฟ — รันบน Google Colab ได้โดยไม่ต้องติดตั้งอะไร

**ขั้นตอน:** โหลดข้อมูล → ดูข้อมูลด้วยตา → ฝึกโมเดล → วัดผล → ดาวน์โหลดโมเดลไปใช้กับ `gateway.py`

## 1. โหลดข้อมูล
ค่าเริ่มต้นใช้ข้อมูลสังเคราะห์ใน repo (`python/data_demo`). ถ้ามีข้อมูลจริงจาก `collect.py` ให้ตั้ง `UPLOAD = True` แล้วอัปโหลด `normal.csv`, `unbalanced.csv`, `off.csv`

In [ ]:
import pandas as pd
UPLOAD = False
REPO_RAW = "https://raw.githubusercontent.com/Boonkon/aiot-motor-guard/main/python/data_demo"  # แก้ Boonkon

if UPLOAD:
    from google.colab import files
    up = files.upload()
    df = pd.concat([pd.read_csv(f) for f in up], ignore_index=True)
else:
    df = pd.concat([pd.read_csv(f"{REPO_RAW}/{s}.csv") for s in ["normal", "unbalanced", "off"]], ignore_index=True)
df["label"].value_counts()

## 2. ดูข้อมูลก่อนฝึก
ถ้า **คนดูกราฟแล้วแยกกลุ่มได้** AI ก็มักแยกได้ ถ้าคนยังแยกไม่ออก อาจต้องเลือกฟีเจอร์ใหม่หรือเก็บข้อมูลเพิ่ม

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for lab, g in df.groupby("label"):
    ax[0].scatter(g["rms"], g["crest"], s=8, label=lab)
    ax[1].hist(g["rms"], bins=40, alpha=.6, label=lab)
ax[0].set(xlabel="rms [g]", ylabel="crest factor"); ax[0].legend()
ax[1].set(xlabel="rms [g]", ylabel="count"); ax[1].legend(); plt.show()

## 3. ฝึกโมเดล Random Forest
แบ่งข้อมูล 75% ไว้ฝึก และ 25% ไว้ทดสอบ (โมเดลไม่เคยเห็น) — **ห้ามวัดผลด้วยข้อมูลที่ใช้ฝึก**

In [ ]:
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

FEATURES = ["rms_x", "rms_y", "rms_z", "rms", "p2p", "crest", "f_dom"]   # ไม่ใช้ temp — ดูบทเรียนที่ 4
X, y = df[FEATURES], df["label"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.25, stratify=y, random_state=0)
clf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=0).fit(X_tr, y_tr)
print(classification_report(y_te, clf.predict(X_te), digits=3))
ConfusionMatrixDisplay.from_estimator(clf, X_te, y_te, cmap="Blues"); plt.show()

## 4. ฟีเจอร์ไหนสำคัญ?

In [ ]:
pd.Series(clf.feature_importances_, index=FEATURES).sort_values().plot.barh(); plt.show()

## 5. ตัวตรวจ "ของแปลก" (unknown)
โมเดลจำแนกตอบได้แค่ 3 คำตอบที่รู้จัก ถ้าเจอรูปแบบใหม่ (เช่น ลูกปืนแตก) มันจะ *เดา* อย่างมั่นใจ — IsolationForest ช่วยบอกว่า "ข้อมูลนี้ไม่เหมือนที่เคยเห็น"

In [ ]:
iso = IsolationForest(random_state=0).fit(X_tr)
thr = float(pd.Series(iso.score_samples(X_tr)).quantile(0.01))
weird = pd.DataFrame([[0.5, 0.5, 0.5, 0.5, 2.0, 1.5, 45]], columns=FEATURES)   # สั่นแรงมาก ความถี่แปลก
print("classifier ทายว่า:", clf.predict(weird)[0])
print("anomaly score:", iso.score_samples(weird)[0].round(3), " threshold:", round(thr, 3), " -> ต่ำกว่า threshold = unknown")

## 6. บันทึกและดาวน์โหลดโมเดล
นำไฟล์ไปวางที่ `python/models/motor_model.joblib` แล้วรัน `gateway.py`

> ⚠️ เวอร์ชัน scikit-learn บน Colab และบน PC ควรตรงกัน (`pip install scikit-learn==<เวอร์ชันที่พิมพ์ด้านล่าง>`)

In [ ]:
import joblib, sklearn
joblib.dump({"clf": clf, "iso": iso, "iso_threshold": thr, "features": FEATURES, "trained_on": "colab"}, "motor_model.joblib")
print("scikit-learn", sklearn.__version__)
try:
    from google.colab import files; files.download("motor_model.joblib")
except ImportError:
    print("บันทึกไว้ที่ motor_model.joblib")